# Bakery Sales Performance Analysis
### End-to-End EDA Pipeline | SQL → Python → Power BI

**Business Problem:** A bakery client had months of sales data but no clarity on which products drove revenue, how customers bought, or how to plan inventory. This notebook delivers the analytical layer of a full pipeline — from raw data to cleaned, insight-ready output for Power BI.

**Dataset:** 141 orders | 27 product columns | Jul–Dec 2019

---
*Tools: Python (pandas, matplotlib, seaborn) | Next step: Power BI dashboard*

## Section 1 — Import Libraries & Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set consistent plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Load the bakery sales dataset
df = pd.read_csv("bakery_clean.csv")
df.tail()

## Section 2 — Data Exploration
Understand the shape, structure, and quality of the dataset before any analysis.

In [ ]:
# Dataset dimensions and column names
print(f"Rows: {df.shape[0]}  |  Columns: {df.shape[1]}")
print(f"\nColumn names:\n{df.columns.tolist()}")

In [ ]:
# Summary statistics — understand ranges and distributions
df.describe()

In [ ]:
# Data types and null counts per column
df.info()

In [ ]:
# Check missing values
df.isnull().sum()

## Section 3 — Data Cleaning
Handle duplicates, missing values, and fix data types before any analysis.

In [ ]:
# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df)}")
print(f"Rows remaining: {len(df)}")

In [ ]:
# Fill nulls with 0 — for product columns, null = 0 units sold
df = df.fillna(0)
print(f"Nulls remaining: {df.isnull().sum().sum()}")

In [ ]:
# Convert product columns (col index 4 onwards) from float to int
# They became float after fillna — int is correct for unit counts
df[df.columns[4:]] = df[df.columns[4:]].astype(int)

# Confirm types are correct
df.dtypes

## Section 4 — Business KPIs
High-level summary metrics — these feed directly into the Power BI dashboard KPI cards.

In [ ]:
total_revenue    = df["Total"].sum()
avg_order_value  = df["Total"].mean()
max_order        = df["Total"].max()
min_order        = df["Total"].min()
total_orders     = len(df)

print("=" * 40)
print("  BAKERY SALES — KEY METRICS")
print("=" * 40)
print(f"  Total Revenue:        ₹{total_revenue:,.0f}")
print(f"  Total Orders:         {total_orders}")
print(f"  Avg Order Value:      ₹{avg_order_value:,.0f}")
print(f"  Highest Single Order: ₹{max_order:,.0f}")
print(f"  Lowest Single Order:  ₹{min_order:,.0f}")
print("=" * 40)

## Section 5 — Product Performance Analysis
Which products sell the most? Which are dead weight?

In [ ]:
# Total units sold per product — sorted highest to lowest
product_sales = df.iloc[:, 4:].sum().sort_values(ascending=False)

print("TOP 5 BEST SELLERS:")
print(product_sales.head(5).to_string())
print()
print("BOTTOM 5 LOWEST SELLERS:")
print(product_sales.tail(5).to_string())

In [ ]:
# Product share of total units sold (%)
product_pct = (product_sales / product_sales.sum() * 100).round(2)
print("Top 10 products by % share of total units:")
print(product_pct.head(10).to_string())

In [ ]:
# Bar chart — all products ranked by units sold
plt.figure(figsize=(10, 7))
sns.barplot(x=product_sales.values, y=product_sales.index, palette="Blues_r")
plt.title("Units Sold by Product", fontsize=13, fontweight="bold")
plt.xlabel("Units Sold")
plt.ylabel("")
plt.tight_layout()
plt.show()
# Finding: Angbutter leads with 192 units — 3x the next best seller (croissant: 82)

## Section 6 — Revenue by Day of Week
Which days drive the most revenue? Informs staffing and stock planning decisions.

In [ ]:
# Total and average revenue per day of week
day_total = df.groupby("Day of Week")["Total"].sum().sort_values(ascending=False)
day_avg   = df.groupby("Day of Week")["Total"].mean().sort_values(ascending=False)

print("Total Revenue by Day:")
print(day_total.to_string())
print()
print("Average Order Value by Day:")
print(day_avg.round(0).to_string())

In [ ]:
# Bar chart — revenue by day
plt.figure(figsize=(8, 5))
sns.barplot(x=day_total.index, y=day_total.values, palette="YlOrBr_r")
plt.title("Total Revenue by Day of Week", fontsize=13, fontweight="bold")
plt.xlabel("Day of Week")
plt.ylabel("Total Revenue (₹)")
plt.tight_layout()
plt.show()
# Finding: Friday is the peak revenue day — optimal for promotions and maximum stock

## Section 7 — Order Value Distribution
How are order values spread? Understanding this helps identify high-value customer segments.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["Total"], bins=20, color="#1B6CA8", edgecolor="white")
plt.title("Distribution of Order Values", fontsize=13, fontweight="bold")
plt.xlabel("Order Value (₹)")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()
# Finding: Most orders cluster in the lower range — a small % of high-value orders
# skew the average upward

## Section 8 — Basket Analysis (Product Co-purchase Patterns)
Which products are frequently bought together? Correlation between product columns reveals bundling opportunities.

In [ ]:
# Correlation matrix across all product columns
corr = df.iloc[:, 4:].corr()

# Heatmap — visual overview of all product correlations
plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", annot=False, linewidths=0.3)
plt.title("Product Co-purchase Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Extract top product pairs ranked by correlation score
# Remove self-correlations and duplicate pairs
pairs = (
    corr.where(~(corr == 1))
        .stack()
        .reset_index()
)
pairs.columns = ["Product 1", "Product 2", "Correlation"]
pairs = pairs[pairs["Product 1"] < pairs["Product 2"]]
top_pairs = pairs.sort_values("Correlation", ascending=False).head(10).round(2)

print("TOP 10 PRODUCT COMBINATIONS (co-purchase correlation):")
print(top_pairs.to_string(index=False))
# Finding: Almond croissant + pandoro (0.52) is the strongest pair — prime bundling candidate

## Section 9 — Inventory Insight
Stock velocity ranking — informs which items need higher stock and which to review or discontinue.

In [ ]:
# Full inventory demand ranking
inventory = df.iloc[:, 4:].sum().sort_values(ascending=False)
print("STOCK VELOCITY RANKING (units sold):")
print(inventory.to_string())

In [ ]:
# Flag dead stock — zero or near-zero sales
dead_stock = inventory[inventory == 0]
low_stock  = inventory[(inventory > 0) & (inventory <= 5)]

print(f"Products with 0 sales (discontinue/reposition): {len(dead_stock)}")
print(dead_stock.to_string())
print()
print(f"Products with 1-5 sales (low performers):")
print(low_stock.to_string())

## Section 10 — Export Cleaned Dataset
Export the cleaned, analysis-ready dataset for import into Power BI.

In [ ]:
# Export to Excel — this file is imported into Power BI for dashboard development
df.to_excel("bakery_clean.xlsx", index=False)
print(f"Exported: bakery_clean.xlsx")
print(f"Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")

---
## Summary of Key Findings

| # | Finding | Detail |
|---|---|---|
| 1 | Top product | Angbutter — 192 units (3x the next best seller) |
| 2 | Peak revenue day | Friday — highest total and average order value |
| 3 | Strongest product pair | Almond croissant + Pandoro — 0.52 correlation |
| 4 | Dead stock | Cheesecake, Croque Monsieur, Mad Garlic, Meringue — 0 sales |
| 5 | Order distribution | Majority of orders are low-value; small % are high-value outliers |

**Next step →** Cleaned dataset (`bakery_clean.xlsx`) imported into Power BI for 3-page executive dashboard development.
